In [ ]:
"""
ECG and PPG Signal Processing and Peak Detection Pipeline

This script implements a complete workflow for processing ECG and PPG signals, 
including the detection and correction of R-Peaks and systolic peaks.

Main functionalities:
- Data loading (.edf and .txt)
- Signal preprocessing
- Peak detection and correction
- Visualization of signals, detected peaks and Heart/Pulse Rate
- Data saving (in .npz format)
"""


# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import numpy as np
import os
import neurokit2 as nk

from utils_signal_processing import (
    load_signals,
    preprocessing,
    samples_to_hms
)
from utils_peaks_correction import (
    peaks_correction, 
    plot_signals_peaks
)
 
%matplotlib qt


# =============================================================================
# DEFINE PARAMETERS AND PATHS
# =============================================================================

# List of NOX subjects
subj_nox = [
    'Pz 501', 'Pz 502', 'Pz 503', 'Pz 504', 'Pz 505', 'Pz 506', 'Pz 507',
    'Pz 508', 'Pz 509', 'Pz 510', 'Pz 511', 'Pz 512', 'Pz 513', 'Pz 514', 
    'Pz 515', 'Pz 516', 'Pz 517', 'Pz 518', 'Pz 532', 'Pz 533', 'Pz 534',
    'Pz 519', 'Pz 520', 'Pz 521', 'Pz 522',
    'Pz 501bis', 'Pz 502bis', 'Pz 503bis', 'Pz 505bis', 'Pz 507bis',
    'Pz 509bis', 'Pz 510bis', 'Pz 511bis', 'Pz 513bis', 'Pz 514bis', 
    'Pz 515bis', 'Pz 516bis', 'Pz 517bis', 'Pz 518bis'
]

# Path to save the final npz files (signals and peaks)
path_save = 'C:/Users/ilari/Desktop/Sleep disorders/Example files/prova elimina' 

# Folder path containing the .edf and .txt files
path_file = 'C:/Users/ilari/Desktop/Sleep disorders/Example files/Database HRV/Pz 249/pz249' 
path_edf = f"{path_file}.edf"
path_hyp = f"{path_file}.txt"


# =============================================================================
# LOAD DATA
# =============================================================================

subj = os.path.basename(path_file)
print(f'--- {subj} Analysis ---')

ecg, ppg, fs_ecg, fs_ppg, unit_ecg, unit_ppg, hyp = load_signals(
    path_edf, path_hyp, subj, subj_nox
)


# =============================================================================
# SIGNAL PREPROCESSING
# =============================================================================

ecg_processed, ppg_processed = preprocessing(
    ecg, ppg, fs_ecg, fs_ppg, unit_ecg, unit_ppg, hyp, subj, subj_nox
)


# =============================================================================
# PEAKS DETECTION
# =============================================================================

# Initial Neurokit peaks (R-peaks in ECG and systolic peaks in PPG)
ecg_peaks, _ = nk.ecg_peaks(
    ecg_processed, sampling_rate=fs_ecg, method='neurokit', show=False
)
ppg_peaks, _ = nk.ppg_peaks(
    ppg_processed, sampling_rate=fs_ppg, method='elgendi', show=False
)

# Neurokit Peaks Correction 
_, ecg_peaks_corrected = nk.signal_fixpeaks(
    ecg_peaks, sampling_rate=fs_ecg, iterative=True, method="Kubios", show=False
)
_, ppg_peaks_corrected = nk.signal_fixpeaks(
    ppg_peaks, sampling_rate=fs_ppg, iterative=True, method="Kubios", show=False
)

# Additional peak corrections
ecg_peaks_final, ppg_peaks_final = peaks_correction(
    ecg_processed, ppg_processed, ecg_peaks_corrected, ppg_peaks_corrected, fs_ecg, fs_ppg
)

# Signal durations 
h_ecg, m_ecg, s_ecg = samples_to_hms(len(ecg_processed), fs_ecg)
h_ppg, m_ppg, s_ppg = samples_to_hms(len(ppg_processed), fs_ppg)
print(f"{h_ecg}h {m_ecg}m {s_ecg}s (ECG), "
      f"{h_ppg}h {m_ppg}m {s_ppg}s (PPG)")


# =============================================================================
# PLOT SIGNALS, DETECTED PEAKS AND HEART/PULSE RATE
# =============================================================================

time_ecg = np.arange(len(ecg_processed)) / fs_ecg  
time_ppg = np.arange(len(ppg_processed)) / fs_ppg 

# Compute HR and PR (for plot)
hr_final = nk.ecg_rate(
    ecg_peaks_final, sampling_rate=fs_ecg, desired_length=len(ecg_processed)
)
pr_final = nk.ppg_rate(
    ppg_peaks_final, sampling_rate=fs_ppg, desired_length=len(ppg_processed)
)

plot_signals_peaks(
    time_ecg, ecg_processed, ecg_peaks_final, time_ppg, ppg_processed, ppg_peaks_final,
    hr_final, pr_final, subj
)


# =============================================================================
# SAVING DATA in npz format
# =============================================================================

path_save_npz = os.path.join(path_save, f"{subj}.npz")
np.savez(
    path_save_npz, ecg=ecg_processed, ppg=ppg_processed,
    ecg_sampling_rate=fs_ecg, ppg_sampling_rate=fs_ppg,
    ecg_peaks=ecg_peaks_final, ppg_peaks=ppg_peaks_final
)
